In [16]:
# Create a copy of dataframes for data cleaning
staging_df = dataframe.copy()

# Checking all data at once
for name, df in staging_df.items():
    print(f'--- {name} ---')
    print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
    print(df.columns.tolist())
    print('\n')

--- contacts ---
Rows: 248397, Columns: 12
['Agent', 'Date', 'LOB', 'Channel', 'ACW', 'Dual/Multiple_Chat_AHT', 'Inbound_Transaction', 'Outbound_Transaction', 'Handle_Time', 'Hold_Time', 'Outbound_Handle_Time', 'Missed_Contacts']


--- productivity ---
Rows: 183860, Columns: 16
['Agent', 'Date', 'Aux_Duration', '1st_BreakDuration', '2nd_Break_Duration', '3rd_Break_Duration', 'Email_Duration', 'Lunch_Duration', 'Meeting_Duration_', 'Technical_Issue_Duration', 'Personal_Duration', 'Task_Duration', 'Training_Duration', 'Available_Duration', 'Busy_Duration', 'Login_Duration']


--- roster ---
Rows: 2544, Columns: 12
['Agent', 'Team Manager', 'Active Date', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Days', 'Tenurity']




In [24]:
# Remove unnecessary columns for table 'roster'
staging_df['roster'] = staging_df['roster'].drop(columns=['Days', 'Tenurity'], errors='ignore')

# Remove columns with NULL values only
staging_df = {name: df.dropna(axis=1, how='all') for name, df in staging_df.items()}

# Standardize column headers for easy access
for name, df in staging_df.items():
    staging_df[name].columns = staging_df[name].columns.str.lower().str.strip().str.replace(r'[^\w]+', '_', regex=True).str.strip('_')

In [25]:
for name, df in staging_df.items():
    print(f'--- {name} ---')
    print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
    print(df.columns.tolist())
    print('\n')

--- contacts ---
Rows: 248397, Columns: 12
['agent', 'date', 'lob', 'channel', 'acw', 'dual_multiple_chat_aht', 'inbound_transaction', 'outbound_transaction', 'handle_time', 'hold_time', 'outbound_handle_time', 'missed_contacts']


--- productivity ---
Rows: 183860, Columns: 16
['agent', 'date', 'aux_duration', '1st_breakduration', '2nd_break_duration', '3rd_break_duration', 'email_duration', 'lunch_duration', 'meeting_duration', 'technical_issue_duration', 'personal_duration', 'task_duration', 'training_duration', 'available_duration', 'busy_duration', 'login_duration']


--- roster ---
Rows: 2544, Columns: 3
['agent', 'team_manager', 'active_date']




In [27]:
# Checking shape and info of the tables
for name, df in staging_df.items():
    staging_df[name].info()
    print()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248397 entries, 0 to 248396
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   agent                   248397 non-null  object
 1   date                    248397 non-null  object
 2   lob                     248397 non-null  object
 3   channel                 248397 non-null  object
 4   acw                     248397 non-null  int64 
 5   dual_multiple_chat_aht  248397 non-null  int64 
 6   inbound_transaction     248397 non-null  int64 
 7   outbound_transaction    248397 non-null  int64 
 8   handle_time             248397 non-null  int64 
 9   hold_time               248397 non-null  int64 
 10  outbound_handle_time    248397 non-null  int64 
 11  missed_contacts         248397 non-null  int64 
dtypes: int64(8), object(4)
memory usage: 22.7+ MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183860 entries, 0 to 183859
Data columns (tot

In [28]:
# Updating data type of `Date` to datetime
date_cols = ['date', 'active_date']
for name, df in staging_df.items():
    for col in date_cols:
        if col in df.columns:
            staging_df[name][col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

In [29]:
for name, df in staging_df.items():
    staging_df[name].info()
    print()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248397 entries, 0 to 248396
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   agent                   248397 non-null  object        
 1   date                    248397 non-null  datetime64[ns]
 2   lob                     248397 non-null  object        
 3   channel                 248397 non-null  object        
 4   acw                     248397 non-null  int64         
 5   dual_multiple_chat_aht  248397 non-null  int64         
 6   inbound_transaction     248397 non-null  int64         
 7   outbound_transaction    248397 non-null  int64         
 8   handle_time             248397 non-null  int64         
 9   hold_time               248397 non-null  int64         
 10  outbound_handle_time    248397 non-null  int64         
 11  missed_contacts         248397 non-null  int64         
dtypes: datetime64[ns](1), int64(8)

In [37]:
# Check exact duplicates
print('Duplicate Counts')
for name, df in staging_df.items():
    dup_counts = df.duplicated(keep=False).sum()
    print()
    print(f'--- {name}: {dup_counts} ---')

Duplicate Counts

--- contacts: 2 ---

--- productivity: 14 ---

--- roster: 0 ---


In [38]:
# Removing exact duplicates
staging2_df = staging_df.copy()
for name, df in staging2_df.items():
    staging2_df[name] = staging2_df[name].drop_duplicates()

print('Duplicate Counts')
for name, df in staging2_df.items():
    dup_counts = df.duplicated(keep=False).sum()
    print()
    print(f'--- {name}: {dup_counts} ---')

Duplicate Counts

--- contacts: 0 ---

--- productivity: 0 ---

--- roster: 0 ---


In [48]:
# 2. Handle logical Duplicates via Aggregation
# For Productivity: One row per Agent per Day
prod_clean = staging2_df['productivity'].groupby(['agent', 'date']).sum().reset_index()
staging2_df['productivity'] = prod_clean

# For Contacts: One row per Agent, Date, LOB, and Channel
# We define our 'Business Key' columns first
key_cols = ['agent', 'date', 'lob', 'channel']
contacts_clean = staging2_df['contacts'].groupby(key_cols).sum().reset_index()
staging2_df['contacts'] = contacts_clean

In [50]:
for name, df in staging_df.items():
    print(name)
    staging_df[name].info()
    print()

for name, df in staging2_df.items():
    print(name)
    staging2_df[name].info()
    print()

contacts
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248397 entries, 0 to 248396
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   agent                   248397 non-null  object        
 1   date                    248397 non-null  datetime64[ns]
 2   lob                     248397 non-null  object        
 3   channel                 248397 non-null  object        
 4   acw                     248397 non-null  int64         
 5   dual_multiple_chat_aht  248397 non-null  int64         
 6   inbound_transaction     248397 non-null  int64         
 7   outbound_transaction    248397 non-null  int64         
 8   handle_time             248397 non-null  int64         
 9   hold_time               248397 non-null  int64         
 10  outbound_handle_time    248397 non-null  int64         
 11  missed_contacts         248397 non-null  int64         
dtypes: datetime64[ns](1),

In [53]:
# Exporting files to csv for MySQL EDA
output_dir = Path.cwd().parent / 'Data' / 'Clean Data'
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in staging2_df.items():
    file_path = output_dir / f'final_{name}.csv'

    # index=False prevents pandas from adding an extra 'unnamed: 0' column
    # date_format ensures MySQL-friendly dates (YYYY-MM-DD)
    df.to_csv(file_path,  index=False, date_format='%Y-%m-%d')

print("Files ready for MySQL import!")

Files ready for MySQL import!
